## Executive Summary

**Purpose:** The part of the stator inductance whose flux crosses the air gap, for any `WindingPattern` on any `StatorGeometry`, and a report of how unequal the phases are.

**What it does:** `winding_function` turns the slot turns into the turns around each tooth. `airgap_inductance` integrates the product of two winding functions over the gap, giving the full m × m phase matrix. It can be split into the working-harmonic part (the magnetising inductance) and the rest (air-gap harmonic, or differential, leakage). `phase_imbalance` reports the spread of the self and mutual inductances and how strongly a positive-sequence current couples into the other sequences.

**Assumptions:** iron of infinite permeability, a smooth non-salient rotor, and a uniform effective gap $g_e = k_C\,g$ (Carter). For a surface-PM rotor pass `g_eff` = $k_C (g + h_m/\mu_r)$. Slot openings are treated as narrow, so the winding function steps at the slot centres; the opening width enters through Carter only. Slot, tooth-tip and end-winding leakage are separate parts, added in the next steps.

## How It Fits Into the Motor Design Workflow

**Upstream:** `emachines.winding.WindingPattern` (turns per slot) and `emachines.inductance.StatorGeometry` (bore, gap, stack, slot opening).

**Downstream:** the total stator inductance (air gap + slot leakage + tooth tip + end winding), dq inductances for motor models, and the comparison of candidate windings.

## Theory

**Winding function.** Tooth *i* lies between slot *i* and slot *i* + 1. Going round the gap, the turns enclosed change by the turns in each slot (Sudhoff's discrete winding function):

$$W_{x,i} = W_{x,i-1} - N_{x,i}, \qquad \sum_i W_{x,i} = 0,$$

with $N_{x,i}$ the turns of phase *x* in slot *i* per ampere of phase current (all positions added, divided by the parallel paths). The zero mean holds because no net flux leaves the rotor. The MMF of phase *x* over tooth *i* is $W_{x,i}\,i_x$.

**Inductance.** With a uniform effective gap $g_e$ at mean radius $r_g = r_{bore} - g/2$, the air-gap flux density over tooth *i* is $\mu_0 \sum_y W_{y,i} i_y / g_e$, and phase *x* links $W_{x,i}$ times the flux through tooth *i*. So

$$L_{ag,xy} = \frac{\mu_0 r_g l_{stk}}{g_e} \int_0^{2\pi} w_x(\theta)\,w_y(\theta)\,d\theta = \frac{\mu_0 r_g l_{stk}}{g_e}\,\frac{2\pi}{Q} \sum_{i=0}^{Q-1} W_{x,i}W_{y,i}.$$

**Harmonic split.** Writing $w_x(\theta) = \sum_{\nu \ge 1} \left(a_{x\nu} e^{j\nu\theta} + \text{c.c.}\right)$, Parseval gives

$$L_{ag,xy} = \frac{\mu_0 r_g l_{stk}}{g_e}\, 4\pi \sum_{\nu \ge 1} \operatorname{Re}\!\left(a_{x\nu}\, a_{y\nu}^*\right).$$

The term $\nu = p$ is the **magnetising** inductance. Everything else, including sub-harmonics of fractional-slot windings, is **air-gap harmonic (differential) leakage**. For a step function the coefficients are exact:

$$a_{x\nu} = \frac{1}{2\pi} \sum_i W_{x,i}\, \frac{e^{-j\nu\theta_i} - e^{-j\nu\theta_{i+1}}}{j\nu}, \qquad \theta_i = \frac{2\pi i}{Q}.$$

**Carter.** $k_C = \tau_s / (\tau_s - \gamma g)$ with $\gamma = (b_0/g)^2 / (5 + b_0/g)$, the same formula as `emachines.motors.calculate_carter_coefficient`, using the slot opening $b_0$ (the full width $b_s$ for an open slot).

## Phase Imbalance

For a balanced m-phase winding the inductance matrix is circulant: every self inductance is equal, and the mutual inductance depends only on how many phases apart two phases are. `phase_imbalance` measures how far a matrix is from that:

- **self spread** $= (\max L_{kk} - \min L_{kk}) / \overline{L_{kk}}$;
- **mutual spread** — the same for each set of mutuals one, two, … phases apart, the largest reported, relative to the mean self inductance;
- **sequence coupling** — transform with the unitary DFT matrix $F_{jk} = e^{-j 2\pi jk/m}/\sqrt{m}$, so $L_{seq} = F L F^H$. A circulant matrix becomes diagonal: $L_{seq,11}$ is the positive-sequence (synchronous) inductance, equal to $L_{aa} - L_{ab}$ for three phases. The largest off-diagonal entry over $|L_{seq,11}|$ says how much flux a balanced positive-sequence current drives into the other sequences. That is what makes unbalanced currents and torque ripple.

Star-of-slots windings come out balanced to rounding error; it is a check of the method. Imbalance appears in hand-made windings (`from_slot_table`, `custom`).

In [ ]:
#| default_exp inductance.airgap

In [ ]:
#| export
from __future__ import annotations

import numpy as np

from emachines.inductance.geometry import StatorGeometry
from emachines.motors.dc_motor import calculate_carter_coefficient
from emachines.winding.pattern import WindingPattern

MU0 = 4e-7 * np.pi

In [ ]:
#| hide
import matplotlib.pyplot as plt
from emachines.inductance.geometry import SlotGeometry
from emachines.winding.pattern import Coil

---

## Winding function

In [ ]:
#| export
def winding_function(pattern: WindingPattern) -> np.ndarray:
    """
    Turns around each tooth per ampere of phase current, zero mean.

    Parameters
    ----------
    pattern : WindingPattern

    Returns
    -------
    np.ndarray, shape (m, Q)
        W[x, i] for tooth i, which lies between slot i and slot i + 1.
    """
    N = pattern.effective_turns().sum(axis=2)          # (m, Q), all positions in a slot
    W = -np.cumsum(N, axis=1)
    return W - W.mean(axis=1, keepdims=True)


def winding_function_harmonics(pattern: WindingPattern, nu) -> np.ndarray:
    """
    Complex Fourier coefficients a_ν of the (step) winding function of each phase.

    w_x(θ) = Σ_ν≥1 (a_xν e^{jνθ} + c.c.). Exact for the step function.

    Parameters
    ----------
    pattern : WindingPattern
    nu      : int or array of int >= 1   Mechanical harmonic orders

    Returns
    -------
    np.ndarray, complex, shape (m,) for scalar nu or (m, len(nu))
    """
    W = winding_function(pattern)
    Q = pattern.Q
    nus = np.atleast_1d(np.asarray(nu, dtype=float))
    if np.any(nus < 1):
        raise ValueError("harmonic orders must be >= 1")
    th = 2 * np.pi * np.arange(Q + 1) / Q
    e = (np.exp(-1j * np.outer(th[:-1], nus)) - np.exp(-1j * np.outer(th[1:], nus))) / (2j * np.pi * nus)
    a = W @ e
    return a[:, 0] if np.ndim(nu) == 0 else a

In [ ]:
# Examples: a full-pitch coil of 10 turns on 12 slots, 2 poles, steps from +5 to −5 turns
pat1 = WindingPattern.custom(12, 2, [Coil(0, 0, "full", 6, "full", 10)], m=1, layout="single")
assert np.allclose(winding_function(pat1), [[-5] * 6 + [5] * 6])
# its fundamental is 4/π of the 5-turn square wave, split over e^{jθ} and e^{-jθ}
assert np.isclose(2 * abs(winding_function_harmonics(pat1, 1)[0]), 4 / np.pi * 5, rtol=2e-2)

---

## Effective air gap and `airgap_inductance`

In [ ]:
#| export
def carter_coefficient(stator: StatorGeometry) -> float:
    """Carter coefficient of the stator slot openings (opening b0, slot pitch at the bore)."""
    return calculate_carter_coefficient(stator.g, stator.slot.b0, stator.tau_s)


def effective_air_gap(stator: StatorGeometry, g_eff: float | None = None) -> float:
    """g_eff if given, else k_C · g."""
    if g_eff is not None:
        if g_eff <= 0:
            raise ValueError(f"g_eff must be positive, got {g_eff}")
        return g_eff
    return carter_coefficient(stator) * stator.g


def airgap_inductance(pattern: WindingPattern, stator: StatorGeometry, part: str = "total",
                      g_eff: float | None = None) -> np.ndarray:
    """
    Air-gap inductance matrix of the phases [H].

    Parameters
    ----------
    pattern : WindingPattern   Winding (its Q must match the stator)
    stator  : StatorGeometry   Bore, gap, stack and slot opening
    part    : str              'total' (default), 'magnetising' (harmonic ν = p only)
                               or 'harmonic' (all other harmonics: differential leakage)
    g_eff   : float | None     Effective gap [m]; None -> Carter-corrected k_C·g.
                               For a surface-PM rotor use k_C (g + h_m / μr).

    Returns
    -------
    np.ndarray, shape (m, m)   L[x, y], symmetric
    """
    if pattern.Q != stator.Q:
        raise ValueError(f"the winding has Q={pattern.Q} slots but the stator has Q={stator.Q}")
    if part not in ("total", "magnetising", "harmonic"):
        raise ValueError(f"part must be 'total', 'magnetising' or 'harmonic', got {part!r}")
    k = MU0 * (stator.r_bore - stator.g / 2) * stator.l_stk / effective_air_gap(stator, g_eff)
    W = winding_function(pattern)
    total = k * (2 * np.pi / pattern.Q) * (W @ W.T)
    if part == "total":
        return total
    a = winding_function_harmonics(pattern, pattern.p)
    mag = k * 4 * np.pi * np.real(np.outer(a, a.conj()))
    return mag if part == "magnetising" else total - mag

---

## `phase_imbalance`

In [ ]:
#| export
def phase_imbalance(L: np.ndarray) -> dict:
    """
    How far a phase inductance matrix is from that of a balanced winding.

    Parameters
    ----------
    L : np.ndarray, shape (m, m)   Phase inductance matrix

    Returns
    -------
    dict with keys:
        self_spread        : float   (max − min) / mean of the self inductances
        mutual_spread      : float   Largest (max − min) among the mutuals k phases apart,
                                     relative to the mean self inductance
        sequence_coupling  : float   Largest off-diagonal |L_seq| / |L_seq[1, 1]|
        L_positive         : float   Positive-sequence (synchronous) inductance, Re L_seq[1, 1]
        L_seq              : np.ndarray   F L F^H with the unitary DFT matrix F
    """
    L = np.asarray(L, dtype=float)
    m = L.shape[0]
    if L.shape != (m, m) or m < 2:
        raise ValueError(f"L must be a square matrix of at least 2 phases, got shape {L.shape}")
    d = np.diag(L)
    mean_self = d.mean()
    self_spread = float((d.max() - d.min()) / mean_self)
    mutual_spread = 0.0
    for k in range(1, m // 2 + 1):
        vals = np.array([L[i, (i + k) % m] for i in range(m)])
        mutual_spread = max(mutual_spread, float((vals.max() - vals.min()) / mean_self))
    j, kk = np.meshgrid(np.arange(m), np.arange(m), indexing="ij")
    F = np.exp(-2j * np.pi * j * kk / m) / np.sqrt(m)
    L_seq = F @ L @ F.conj().T
    off = L_seq - np.diag(np.diag(L_seq))
    L_pos = L_seq[1, 1]
    return dict(self_spread=self_spread, mutual_spread=mutual_spread,
                sequence_coupling=float(np.abs(off).max() / abs(L_pos)),
                L_positive=float(L_pos.real), L_seq=L_seq)

---

## Examples

The 36-slot stator from the geometry notebook with a short-pitched double-layer winding, 10 turns per coil, 2 parallel paths:

In [ ]:
stator = StatorGeometry(Q=36, r_bore=60e-3, g=0.8e-3, l_stk=100e-3, h_y=12e-3, l_ew=22e-3,
                        slot=SlotGeometry.semi_closed(b0=2.5e-3, h0=1e-3, h1=1.5e-3, bs=6e-3, hs=18e-3, t=0.5))
pat = WindingPattern.distributed(36, 4, pitch=7, turns=10, parallel_paths=2)
L = airgap_inductance(pat, stator)
Lm = airgap_inductance(pat, stator, "magnetising")
Lh = airgap_inductance(pat, stator, "harmonic")
print(f"Carter k_C = {carter_coefficient(stator):.4f}, g_e = {effective_air_gap(stator) * 1e3:.4f} mm")
print(f"L_aa = {L[0, 0] * 1e3:.4f} mH, L_ab = {L[0, 1] * 1e3:.4f} mH")
print(f"positive sequence: magnetising {phase_imbalance(Lm)['L_positive'] * 1e3:.4f} mH "
      f"+ harmonic leakage {phase_imbalance(Lh)['L_positive'] * 1e3:.4f} mH")
assert np.allclose(L, Lm + Lh) and np.allclose(L, L.T)

The winding function of phase A under its slot diagram, with the fundamental that gives the magnetising part:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5), gridspec_kw=dict(height_ratios=[1, 1.3]))
pat.plot(ax=axes[0])
W = winding_function(pat)[0]
x = np.arange(pat.Q + 1)
axes[1].step(x + 0.4, np.append(W, W[0]), where="post", color="C0", label="W_A (step, per tooth)")
th = np.linspace(0, 2 * np.pi, 721)
a = winding_function_harmonics(pat, pat.p)[0]
axes[1].plot(th * pat.Q / (2 * np.pi) + 0.4, 2 * np.real(a * np.exp(1j * pat.p * th)), "k--", lw=1,
             label=f"fundamental ν = p = {pat.p}")
axes[1].set_xlim(-0.3, pat.Q + 0.1); axes[1].set_xlabel("slot"); axes[1].set_ylabel("turns per A of phase current")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()

A hand-made winding that is **not** balanced: the same table as a balanced 12-slot, 2-pole double layer, but with one phase-A coil side moved into the neighbouring slot. The report shows the imbalance, and the air gap alone already sees it:

In [ ]:
table = np.array([[1, 1, -3, -3, 2, 2, -1, -1, 3, 3, -2, -2],
                  [1, -3, -3, 2, 2, -1, -1, 3, 3, -2, -2, 1]])
st12 = StatorGeometry(Q=12, r_bore=40e-3, g=0.6e-3, l_stk=60e-3, h_y=10e-3, slot=SlotGeometry.open(8e-3, 15e-3))
ok = WindingPattern.from_slot_table(table, P=2, turns=20)
bad_table = table.copy(); bad_table[1, 0], bad_table[1, 1] = bad_table[1, 1], bad_table[1, 0]   # swap A and −C in the bottom layer
bad = WindingPattern.from_slot_table(bad_table, P=2, turns=20)
for name, p in [("balanced", ok), ("one side moved", bad)]:
    r = phase_imbalance(airgap_inductance(p, st12))
    print(f"{name:15s} self spread {100 * r['self_spread']:6.3f} %, mutual spread {100 * r['mutual_spread']:6.3f} %, "
          f"sequence coupling {100 * r['sequence_coupling']:6.3f} %")
assert phase_imbalance(airgap_inductance(ok, st12))["sequence_coupling"] < 1e-12
assert phase_imbalance(airgap_inductance(bad, st12))["sequence_coupling"] > 0.01

---

## Validation

1. **Magnetising inductance against the closed form.** For any balanced winding the working-harmonic self inductance is $L_m = \dfrac{4}{\pi}\,\dfrac{\mu_0 r_g l_{stk}}{g_e}\left(\dfrac{k_{w1} N_s}{p}\right)^2$, with mutuals $L_m \cos(2\pi k/m)$, so $-L_m/2$ for three phases (Krause, Wasynczuk & Sudhoff; Pyrhönen et al.). Here $k_{w1}$ and $N_s$ come from the star-of-slots phasors (`WindingPattern.winding_factor`, `series_turns`), an independent route from the winding function.
2. **Air-gap harmonic leakage of full-pitch integer-slot windings.** For three phases, $\sigma_\delta = L_{pos}/L_{pos,1} - 1 = \dfrac{\pi^2 (5q^2 + 1)}{54\, q^2 k_{w1}^2} - 1$, which gives the classical tabulated values 0.0966, 0.0284, 0.0141, 0.0089 for q = 1 … 4 (Richter; Pyrhönen et al.).
3. **Parseval.** The harmonic sum converges to the direct integral.

In [ ]:
# 1. magnetising inductance = closed form, for distributed, fractional-slot and tooth-coil windings
cases = [(36, 4, dict(kind="distributed", pitch=7)), (24, 4, dict(kind="distributed", pitch=6)),
         (24, 4, dict(kind="distributed", layers=1)), (36, 8, dict(kind="sos")), (12, 10, dict(kind="concentrated")),
         (12, 10, dict(kind="concentrated", all_teeth=False)), (9, 8, dict(kind="concentrated", layout="over_under"))]
for Q, P, kw in cases:
    kind = kw.pop("kind")
    p_ = {"distributed": WindingPattern.distributed, "sos": WindingPattern.from_star_of_slots,
          "concentrated": WindingPattern.concentrated}[kind](Q, P, turns=7, **kw)
    st = StatorGeometry(Q=Q, r_bore=55e-3, g=0.7e-3, l_stk=80e-3, h_y=10e-3, slot=SlotGeometry.open(3e-3, 15e-3))
    Lm_ = airgap_inductance(p_, st, "magnetising")
    k = MU0 * (st.r_bore - st.g / 2) * st.l_stk / effective_air_gap(st)
    closed = 4 / np.pi * k * (p_.winding_factor()[0] * p_.series_turns()[0] / p_.p) ** 2
    assert np.allclose(np.diag(Lm_), closed, rtol=1e-10), (Q, P, kind)
    assert np.allclose(Lm_[0, 1], closed * np.cos(2 * np.pi / 3), rtol=1e-10), (Q, P, kind)
    print(f"{Q:2d}s/{P:2d}p {kind:12s}: L_m = {closed * 1e3:8.4f} mH (closed form) — matched")

# 2. differential leakage of full-pitch integer-slot windings
for q, tab in zip(range(1, 6), [0.0966, 0.0284, 0.0141, 0.0089, 0.0065]):
    p_ = WindingPattern.distributed(6 * q, 2)
    st = StatorGeometry(Q=6 * q, r_bore=55e-3, g=0.7e-3, l_stk=80e-3, h_y=10e-3, slot=SlotGeometry.open(2e-3, 15e-3))
    sigma = phase_imbalance(airgap_inductance(p_, st))["L_positive"] / phase_imbalance(airgap_inductance(p_, st, "magnetising"))["L_positive"] - 1
    closed = np.pi ** 2 * (5 * q ** 2 + 1) / (54 * q ** 2 * p_.winding_factor()[0] ** 2) - 1
    assert np.isclose(sigma, closed, rtol=1e-9) and np.isclose(sigma, tab, atol=5e-5), (q, sigma)
    print(f"q = {q}: σ_δ = {sigma:.4f} (tabulated {tab})")

Tooth-coil windings carry far more harmonic leakage than distributed ones, which is why their synchronous inductance is high:

In [ ]:
for Q, P in [(12, 10), (9, 8), (12, 8), (24, 22)]:
    p_ = WindingPattern.concentrated(Q, P)
    st = StatorGeometry(Q=Q, r_bore=55e-3, g=0.7e-3, l_stk=80e-3, h_y=10e-3, slot=SlotGeometry.open(3e-3, 15e-3))
    s = phase_imbalance(airgap_inductance(p_, st))["L_positive"] / phase_imbalance(airgap_inductance(p_, st, "magnetising"))["L_positive"] - 1
    print(f"{Q}s/{P}p all teeth: σ_δ = {s:.3f}")

---

## Tests

In [ ]:
#| hide
from dataclasses import replace

# Parseval: the harmonic sum converges to the direct integral (step function: error ~ 1/ν_max)
p_ = WindingPattern.distributed(24, 4, pitch=5, turns=3)
a = winding_function_harmonics(p_, np.arange(1, 20001))
assert np.allclose(4 * np.pi * np.real(a @ a.conj().T), (2 * np.pi / 24) * winding_function(p_) @ winding_function(p_).T,
                   rtol=1e-4, atol=1e-6)
# scalar and vector harmonic calls agree
assert np.allclose(winding_function_harmonics(p_, 2), winding_function_harmonics(p_, [2])[:, 0])

# balanced windings: zero mean winding function, circulant matrix, no imbalance
for p_ in [WindingPattern.distributed(36, 4, pitch=7), WindingPattern.concentrated(12, 10),
           WindingPattern.concentrated(12, 10, all_teeth=False), WindingPattern.from_star_of_slots(15, 8)]:
    st = StatorGeometry(Q=p_.Q, r_bore=55e-3, g=0.7e-3, l_stk=80e-3, h_y=10e-3, slot=SlotGeometry.open(3e-3, 15e-3))
    assert np.allclose(winding_function(p_).mean(axis=1), 0)
    r = phase_imbalance(airgap_inductance(p_, st))
    assert r["self_spread"] < 1e-12 and r["mutual_spread"] < 1e-12 and r["sequence_coupling"] < 1e-12, p_.name

# scaling: L ∝ N², 1/a², l_stk, 1/g_eff; parallel paths
st = StatorGeometry(Q=24, r_bore=55e-3, g=0.7e-3, l_stk=80e-3, h_y=10e-3, slot=SlotGeometry.open(3e-3, 15e-3))
L1 = airgap_inductance(WindingPattern.distributed(24, 4, pitch=5), st)
assert np.allclose(airgap_inductance(WindingPattern.distributed(24, 4, pitch=5, turns=4, parallel_paths=2), st), 4 * L1)
assert np.allclose(airgap_inductance(WindingPattern.distributed(24, 4, pitch=5), replace(st, l_stk=160e-3)), 2 * L1)
ge = effective_air_gap(st)
assert np.allclose(airgap_inductance(WindingPattern.distributed(24, 4, pitch=5), st, g_eff=2 * ge), L1 / 2)

# Carter: a narrower opening gives a smaller k_C; an open slot uses bs
sc = replace(st, slot=SlotGeometry.semi_closed(1.5e-3, 1e-3, 1e-3, 3e-3, 15e-3))
assert 1 < carter_coefficient(sc) < carter_coefficient(st)
assert np.isclose(carter_coefficient(st), calculate_carter_coefficient(0.7e-3, 3e-3, st.tau_s))

# phase_imbalance on a hand-made circulant and a perturbed matrix
C = np.array([[3.0, -1.0, -1.0], [-1.0, 3.0, -1.0], [-1.0, -1.0, 3.0]])
r = phase_imbalance(C)
assert np.isclose(r["L_positive"], 4.0) and r["sequence_coupling"] < 1e-12
C2 = C.copy(); C2[0, 0] = 3.3
r2 = phase_imbalance(C2)
assert np.isclose(r2["self_spread"], 0.3 / C2.diagonal().mean()) and r2["sequence_coupling"] > 0.02

# errors
for f in [lambda: airgap_inductance(WindingPattern.distributed(36, 4), st), lambda: airgap_inductance(p_, st, "slot"),
          lambda: effective_air_gap(st, 0.0), lambda: winding_function_harmonics(p_, 0), lambda: phase_imbalance(np.ones((1, 1)))]:
    try:
        f(); raise AssertionError("expected ValueError")
    except ValueError:
        pass

## References

- Sudhoff, S.D. *Power Magnetic Devices* (2nd ed.), §8.1–8.5, and [*Distributed Windings and Rotating Fields*](https://engineering.purdue.edu/~sudhoff/ee595s/TADES-chapter4.pdf), Purdue notes, §4.3–4.6 — discrete winding function, air-gap inductance.
- Krause, P.C., Wasynczuk, O., Sudhoff, S.D., Pekarek, S. (2013). *Analysis of Electric Machinery and Drive Systems* (3rd ed.). Wiley-IEEE Press — magnetising inductance of sinusoidally distributed windings.
- Pyrhönen, J., Jokinen, T., Hrabovcová, V. (2014). *Design of Rotating Electrical Machines* (2nd ed.). Wiley — Carter factor, air-gap (harmonic) leakage factor σ_δ.
- Scuiller, F., Semail, E., Charpentier, J.-F. (2010). [General modeling of the windings for multi-phase ac machines](https://www.epjap.org/articles/epjap/pdf/2010/06/ap09444.pdf). *Eur. Phys. J. Appl. Phys.* 50:31102 — air-gap inductance matrix as a product of sampled winding functions.